# Cierre de Fase 5 — síntesis

Fase 5 se pidió con 8 preguntas explícitas, cubriendo desde la comparación de modelos hasta una
predicción real de la próxima semana. Este notebook responde cada una directamente, con la
referencia al notebook que la respalda — no repite el detalle, lo sintetiza.

## 1. ¿Mejores resultados que lo ya reportado?

**Sí**, medido de la forma más concreta posible: las mismas 929 predicciones reales que el
pipeline heredado ya guardó para las semanas 7-18 de 2025, comparadas contra el resultado real de
esas semanas (`16_benchmark_vs_legado.ipynb`):

| Target | MAE legado | MAE nuevo | Mejora | R² legado | R² nuevo |
|---|---|---|---|---|---|
| `receptions` | 1.722 | 1.644 | 4.5% | 0.219 | **0.260** |
| `receiving_yards` | 27.049 | 25.113 | 7.2% | 0.123 | **0.205** |
| `receiving_tds` | 0.412 | 0.341 | **17.2%** | -0.044 | -0.097 |

En `receiving_tds` el MAE mejora con claridad, pero el R² del modelo nuevo es más negativo que el
del legado -- la misma tensión MAE-vs-R² ya documentada en `14_receiving_tds_modelos_de_conteo.ipynb`
y confirmada aquí de forma independiente, no oculta.

## 2. ¿Estables en el tiempo?

**Sí**, con evidencia de 3 fuentes independientes, no de un solo split:
- Walk-forward de 5 años (2021-2025): coeficiente de variación de 4.5%-4.7% en los 3 targets
  (`15_estabilidad_temporal.ipynb`) -- el desempeño no depende de qué año le tocó de prueba.
- Corte por semana dentro de temporada: el MAE se mueve poco entre semanas 1-4 y 14-18 -- no hace
  falta medio año de forma acumulada para que el modelo funcione bien (`15_estabilidad_temporal.ipynb`).
- Validación contra 2026, temporada en curso, dato que no existía en ningún momento anterior del
  proyecto: MAE en el mismo rango que el test 2024-2025, sin degradarse (`18_validacion_temporada_actual.ipynb`).

## 3. ¿Tienen sentido?

**Sí** -- el ranking de la predicción real de la próxima semana (`18_validacion_temporada_actual.ipynb`)
son, de principio a fin, receptores WR1 reconocibles (Jaxon Smith-Njigba, Amon-Ra St. Brown,
Ja'Marr Chase, Puka Nacua, CeeDee Lamb, Justin Jefferson...), no nombres aleatorios. El caso
individual de Chase en `17_explicabilidad_shap.ipynb` también es coherente con lo que se sabe de
él, aunque el modelo se haya quedado corto esa semana en particular (ver punto 6).

## 4. ¿Se respetan las suposiciones de cada modelo?

- **Ridge**: se agregó escalado (`StandardScaler`), corrigiendo al patrón heredado que solo
  imputaba -- el MAE apenas cambió, pero la interpretación de los coeficientes sí, de forma
  sustancial (`13_comparacion_modelos.ipynb`, sección 1).
- **RandomForest**: se imputan los nulos con la mediana de entrenamiento -- `sklearn` no acepta
  `NaN` en este modelo, a diferencia de XGBoost/HistGradientBoosting.
- **`receiving_tds`**: se usó un objetivo `reg:tweedie` (enlace logarítmico), respetando que es un
  conteo no negativo con exceso de ceros -- no una regresión genérica de error cuadrático como
  hizo el pipeline heredado (que por eso tuvo R² negativo, ver punto 1).

## 5. ¿Son explicables?

**Sí** -- SHAP (`17_explicabilidad_shap.ipynb`) sobre los 3 modelos ganadores (los 3 son XGBoost),
con 3 métodos de importancia distintos (permutación en Fase 4, importancia nativa de árbol en
`13_comparacion_modelos.ipynb`, SHAP aquí) apuntando a las mismas variables. Se documentó
explícitamente que las variables SHAP de `receiving_tds` se leen en escala logarítmica
(multiplicativa), no aditiva -- una lectura incorrecta ahí habría sido un error real de
interpretación, no solo un detalle técnico.

## 6. Competencia real de modelos, y por qué gana el ganador

Baseline / Ridge / RandomForest / XGBoost / HistGradientBoosting compitieron en los 3 targets
(`13_comparacion_modelos.ipynb`, `14_receiving_tds_modelos_de_conteo.ipynb`). **XGBoost gana en
los 3** (con objetivo `reg:tweedie` en `receiving_tds`). No solo por el MAE más bajo: su
importancia de variables coincide con lo que Fase 4 ya había señalado (`target_share`/`wopr`
recientes, `depth_team`), y el análisis de error por segmento de uso reveló un sesgo honesto y
consistente en los 3 targets con árboles: **el modelo subestima sistemáticamente las semanas
boom** -- confirmado 3 veces de forma independiente (cuartil de uso en `13`, caso real de Chase y
los 5 peores errores en `17`). Es el mismo patrón que ya documentó Daniel en su propia exploración
previa (`preeliminar/03_modelo_predictivo/05_resultados_y_trazabilidad.ipynb`) -- no es un defecto
de este proyecto en particular, es un patrón estructural del problema.

## 7. ¿Hay otras técnicas mejores, y se ejecutaron?

**Sí, para `receiving_tds`**: se probaron y ejecutaron de verdad (no solo se mencionaron) Poisson,
Tweedie, XGBoost con objetivo de conteo, HistGradientBoosting con pérdida Poisson, y un modelo de
2 etapas (*hurdle*) -- `14_receiving_tds_modelos_de_conteo.ipynb`. `XGBoost (reg:tweedie)` ganó por
MAE con una ventaja clara (0.247 vs. 0.307-0.312 del resto), aunque con la salvedad de R² ya
documentada. Para `receptions`/`receiving_yards` los modelos genéricos ya capturaban bien la señal
-- no hacía falta una técnica especializada ahí.

## 8. Validado con datos recientes, y aplicado a predecir lo que viene

`18_validacion_temporada_actual.ipynb`: el modelo entrenado con 2016-2025 se validó contra las
semanas ya jugadas de 2026 (MAE en el mismo rango que el test histórico) y generó una predicción
real para la semana 4 de 2026, todavía sin jugarse, con su fecha de frescura de datos documentada.
Es una demostración de una sola corrida -- la infraestructura recurrente de producción sigue
siendo trabajo de Fase 6/7.

## Decisión de Fase 4 resuelta con evidencia

Pospuesta a propósito en Fase 4: ¿entrenar con todas las semanas o solo con las activas
(`targets > 0`)? Resuelto en `13` y `14`, con la comparación metodológicamente correcta (mismo
conjunto de prueba activo en ambos casos): **entrenar con todas las semanas gana en los 3 targets
y en los 5 modelos** -- las semanas en cero no son ruido, siguen aportando contexto real del
jugador y su equipo.

## Tarjeta-resumen por target (ligera -- no reemplaza el Model Card formal de Fase 6/7)

In [1]:
import pandas as pd
pd.set_option("display.width", 160)

tarjetas = pd.DataFrame({
    "receptions": {
        "modelo": "XGBoost (max_depth=4, n_estimators=200, lr=0.03)",
        "mae_test_2024_2025": 1.372, "mae_vs_legado_2025_real": "1.644 vs 1.722 (-4.5%)",
        "r2_test": 0.445, "estabilidad_cv": "4.7%", "caveat": "ninguno mayor",
    },
    "receiving_yards": {
        "modelo": "XGBoost (max_depth=3, n_estimators=200, lr=0.03)",
        "mae_test_2024_2025": 20.647, "mae_vs_legado_2025_real": "25.113 vs 27.049 (-7.2%)",
        "r2_test": 0.363, "estabilidad_cv": "4.5%", "caveat": "subestima semanas boom (ver punto 6)",
    },
    "receiving_tds": {
        "modelo": "XGBoost (objective=reg:tweedie, power=1.5)",
        "mae_test_2024_2025": 0.2466, "mae_vs_legado_2025_real": "0.341 vs 0.412 (-17.2%)",
        "r2_test": -0.045, "estabilidad_cv": "4.6%",
        "caveat": "R2 casi igual al baseline pese a mejor MAE -- target estructuralmente difícil (82% en cero)",
    },
}).T
tarjetas

,modelo,mae_test_2024_2025,mae_vs_legado_2025_real,r2_test,estabilidad_cv,caveat
receptions,"XGBoost (max_depth=4, n_estimators=200, lr=0.03)",1.372,1.644 vs 1.722 (-4.5%),0.445,4.7%,ninguno mayor
receiving_yards,"XGBoost (max_depth=3, n_estimators=200, lr=0.03)",20.647,25.113 vs 27.049 (-7.2%),0.363,4.5%,subestima semanas boom (ver punto 6)
receiving_tds,"XGBoost (objective=reg:tweedie, power=1.5)",0.2466,0.341 vs 0.412 (-17.2%),-0.045,4.6%,R2 casi igual al baseline pese a mejor MAE -- ...


## Alcance diferido explícitamente (no en silencio)

- **Model Cards formales** (`weekly/wr/models/*.md`): esta ronda fue "primeras pruebas" -- una
  Model Card de producción necesita el modelo ya serializado con criterio de servir predicciones.
- **`tracking.csv`**: pospuesto a Fase 6, su entregable declarado -- no tiene sentido versionar
  corridas antes de tener un modelo elegido con esta confianza.
- **Regresión de cuantiles (P10/P50/P90)**: pospuesta a un seguimiento corto posterior -- no era
  parte de las 8 preguntas explícitas de esta ronda, y agregarla ahora habría diluido rigor en lo
  que sí se pidió. Técnicamente sencilla de retomar
  (`HistGradientBoostingRegressor(loss="quantile", quantile=0.1/0.9)` sobre el mismo pipeline).
- **Unificación de `depth_charts` 2024→2025**: `depth_team` sale 100% nulo para 2025 en adelante
  (confirmado otra vez en `18_validacion_temporada_actual.ipynb`) -- limitación conocida desde
  Fase 2, ahora con más evidencia de que vale la pena resolverla (es una de las variables más
  importantes del modelo).

## Qué sigue

Con un modelo ganador por target, validado, explicado y probado contra datos reales de hoy, Fase 6
(estabilidad/ciclo de vida: `tracking.csv`, checklist de calidad recurrente) y Fase 7 (documentar
el patrón para que Ricky lo replique en otras posiciones) son los siguientes pasos naturales del
plan original -- a definir con Daniel cuándo arrancar.

Notebook anterior: [`18_validacion_temporada_actual.ipynb`](18_validacion_temporada_actual.ipynb) — último de la Fase 5.